In [1]:
import os 
os.chdir("../")
%pwd

'd:\\Programming\\ML\\End-to-End\\End-to-End-TelcoChurn'

In [2]:
from dataclasses import dataclass 
from pathlib import Path 

@dataclass(frozen=True)
class DataTransformationConfig:
    root_dir: Path 
    data_path: Path
    transformer_path: Path
    target_name: str
    X_train_path: Path 
    X_test_path: Path 
    y_train_path: Path 
    y_test_path: Path

In [3]:
from src.constants import *
from src.utils import read_yaml, create_directories

class ConfigurationManager:
    def __init__(self,
                config_path = CONFIG_FILE_PATH,
                params_path = PARAMS_FILE_PATH,
                schema_path = SCHEMA_FILE_PATH):
        self.config = read_yaml(config_path)
        self.params = read_yaml(params_path)
        self.schema = read_yaml(schema_path)
        
        create_directories([self.config.artifacts_root])
        
        
    def get_data_transformation_config(self)-> DataTransformationConfig:
        config = self.config.data_transformation
        create_directories([config.root_dir])
        
        return DataTransformationConfig(
            root_dir= Path(config.root_dir),
            data_path=Path(config.data_path),
            transformer_path=Path(config.transformer_path),
            target_name= config.target_name,
            X_train_path=Path(config.X_train_path),
            X_test_path=Path(config.X_test_path),
            y_train_path=Path(config.y_train_path),
            y_test_path=Path(config.y_test_path)
        )

In [ ]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.model_selection import train_test_split
import joblib
from src import logging
from scipy import sparse

class DataTransformation:
    def __init__(self, config: DataTransformationConfig):
        self.config = config

    def get_data_transformer_object(self, df):
        logging.info("Getting data transformer object dynamically")

        # --- Step 1: Ensure TotalCharges is numeric ---
        if 'TotalCharges' in df.columns:
            df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

        # --- Step 2: Identify columns ---
        numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
        object_cols = df.select_dtypes(include=['object']).columns.tolist()

        # --- Step 3: Classify categorical columns ---
        binary_cols = [col for col in object_cols if df[col].nunique() == 2]
        multi_category_cols = [col for col in object_cols if df[col].nunique() > 2]

        logging.info(f"Numerical cols: {numerical_cols}")
        logging.info(f"Binary cols: {binary_cols}")
        logging.info(f"Multi-category cols: {multi_category_cols}")

        # --- Step 4: Build transformers ---
        numeric_transformer = Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ])

        binary_transformer = Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OrdinalEncoder())
        ])

        multi_cat_transformer = Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OneHotEncoder(handle_unknown='ignore'))
        ])

        # --- Step 5: Combine everything ---
        return ColumnTransformer(transformers=[
            ('num', numeric_transformer, numerical_cols),
            ('bin', binary_transformer, binary_cols),
            ('multi', multi_cat_transformer, multi_category_cols)
        ])
    
    def initiate_data_transformer(self):
        logging.info("Initiating data transformer")

        df = pd.read_csv(self.config.data_path)
        target_name = str(self.config.target_name)

        X = df.drop(columns=target_name)
        y = df[target_name]

        preprocessing_obj = self.get_data_transformer_object(X)

        # --- Split data ---
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

        # --- Encode target variable ---
        y_train = y_train.map({'Yes': 1, 'No': 0}).fillna(0).astype(int)
        y_test = y_test.map({'Yes': 1, 'No': 0}).fillna(0).astype(int)

        logging.info(f"\nX_train: {X_train.shape}\nX_test: {X_test.shape}\ny_train: {y_train.shape}\ny_test: {y_test.shape}")

        # --- Apply preprocessing ---
        logging.info("Applying preprocessing on training and testing data")
        X_train_arr = preprocessing_obj.fit_transform(X_train)
        X_test_arr = preprocessing_obj.transform(X_test)

        # --- Save preprocessing object ---
        with open(self.config.transformer_path, "wb") as f:
            joblib.dump(preprocessing_obj, f)

        # --- save splits ---
        # there is two way to save sparse (scipy , joblib)
        sparse.save_npz(file=self.config.X_train_path, matrix=X_train_arr) # .npz
        joblib.dump(value=X_test_arr, filename=self.config.X_test_path)    # .pkl
        
        y_train.to_csv(self.config.y_train_path, index=False)
        y_test.to_csv(self.config.y_test_path, index=False)

        return preprocessing_obj, X_train_arr, X_test_arr, y_train, y_test


ModuleNotFoundError: No module named 'src'

In [5]:
try:
    config = ConfigurationManager()
    data_transformation_config = config.get_data_transformation_config()
    data_transformation = DataTransformation(config=data_transformation_config)
    data_transformation.initiate_data_transformer()
except Exception as e:
    raise e

[2025-10-16 07:32:35,162] [INFO] [root:read_yaml:16] - reading the content of 'config\config.yaml'
[2025-10-16 07:32:35,166] [INFO] [root:read_yaml:16] - reading the content of 'params.yaml'
[2025-10-16 07:32:35,170] [INFO] [root:read_yaml:16] - reading the content of 'schema.yaml'
[2025-10-16 07:32:35,172] [INFO] [root:create_directories:39] - created directory at: artifacts
[2025-10-16 07:32:35,174] [INFO] [root:create_directories:39] - created directory at: artifacts/data_transformation
[2025-10-16 07:32:35,174] [INFO] [root:initiate_data_transformer:60] - Initiating data transformer


[2025-10-16 07:32:35,221] [INFO] [root:get_data_transformer_object:16] - Getting data transformer object dynamically
[2025-10-16 07:32:35,252] [INFO] [root:get_data_transformer_object:30] - Numerical cols: ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']
[2025-10-16 07:32:35,253] [INFO] [root:get_data_transformer_object:31] - Binary cols: ['gender', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']
[2025-10-16 07:32:35,254] [INFO] [root:get_data_transformer_object:32] - Multi-category cols: ['customerID', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaymentMethod']
[2025-10-16 07:32:35,263] [INFO] [root:initiate_data_transformer:77] - 
X_train: (5634, 20)
X_test: (1409, 20)
y_train: (5634,)
y_test: (1409,)
[2025-10-16 07:32:35,264] [INFO] [root:initiate_data_transformer:80] - Applying preprocessing on training and testing data
